# Assignment 2

The goal of this assignment is to design and implement an AI system with a conversational interface.

Before you begin, keep in mind that meeting the requirements is important, but more important is that you solve the technical problems associated with the implementation. The assignment is fairly open-ended and can easily become an expansive project. My recommendation is that you implement a simplified version of the services, before moving to more complex implementation. Remember to test your code constantly.  

# Requirements

Your project should meet the following specifications.

## Services

You must include at least **three services** in your system.

### Service 1: API Calls

* One service must use an API as its back end.
* You can refer to the list of [public and free APIs on GitHub](https://github.com/public-apis/public-apis).
* This service may simply return the API’s output to the user, but the response must not be provided verbatim. Instead, transform or rephrase the output, for example, by summarizing, rewriting in a natural tone, or converting structured data into written text.

## Guardrails and Other Limitations

* Include guardrails that prevent users from:

  * Accessing or revealing the system prompt.
  * Modifying the system prompt directly.

* The model must not respond to questions on certain restricted topics:

  * Cats or dogs
  * Horoscopes or Zodiac Signs
  * Taylor Swift

## Implementation

+ Implement your code in the folder `./05_src/assignment_chat`.
+ Add a `readme.md` where you explain the nature of your chat client, the serivices that it provides, and any decisions that you made related to the implementation.
+ We will not be able to install more libraries to assess your work. Please use the standard setup of the course.

# Submission Information

**Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/deploying-ai/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.


In [11]:
#load keys
%load_ext dotenv
%dotenv ../.secrets

#get location#
import os
os.getcwd() #assignment_chat

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


'/Users/benho/Desktop/GoogleDrive_BS/Work/OtherWork/UT_DSI/ai/deploying-ai/05_src/assignment_chat'

In [17]:
##Use News API##
#set environment
import requests, json
import os
from openai import OpenAI
from dotenv import load_dotenv

# Load environment variables
load_dotenv()
load_dotenv(".secrets")

#control number of articles
num_articles=3 # 3 max for free tier

API_TOKEN = os.getenv("THENEWSAPI_API_TOKEN")
URL = f"https://api.thenewsapi.com/v1/news/top?api_token={API_TOKEN}&locale=us&language=en&limit={num_articles}"

response = requests.get(URL)
data = response.json()
print(json.dumps(data, indent=2))

if "data" in data:
    for article in data["data"]:
        #print(article["snippet"])
        print()
else:
    print("⚠️ Unexpected response structure.")


{
  "meta": {
    "found": 1475457,
    "returned": 3,
    "limit": 3,
    "page": 1
  },
  "data": [
    {
      "uuid": "948be2f1-bb49-4a84-b931-5cbf24667b10",
      "title": "JTN Networks Acquires Post Millennial And Human Events In Union Of Brands On The Right",
      "description": "JTN Networks, the parent company of Just the News, has acquired the Human Events and The Post Millennial, uniting news brands on the right.",
      "keywords": "",
      "snippet": "JTN Networks, the parent company of Just the News, has acquired the Human Events and The Post Millennial, uniting news brands on the right.\n\nHuman Events, which...",
      "url": "https://deadline.com/2025/11/right-wing-media-brands-merge-just-the-news-human-events-1236616651/",
      "image_url": "https://deadline.com/wp-content/uploads/2025/11/GettyImages-2244546604.jpg?w=1024",
      "language": "en",
      "published_at": "2025-11-14T17:59:00.000000Z",
      "source": "deadline.com",
      "categories": [
        "ent

In [18]:
#set up OpenAI#

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
if not OPENAI_API_KEY:
    raise ValueError("Please set the OPENAI_API_KEY environment variable.")

client = OpenAI(api_key=OPENAI_API_KEY)

In [30]:
##Define output and set up prompt##
#define output 
from pydantic import BaseModel
import re

class SummaryOutput(BaseModel):
    UUID: str
    Title: str
    Subtitle: str
    Keywords: str
    Summary: str
    Language: str
    Source: str
    URL: str
    Categories: str
    Score: int
    Reason: str
    Input_tokens: int
    Output_tokens: int

#functions

#Service 1 - get news via API #
def ask_chatgpt(document_text: str):
    """Summarise the provided news for children"""
    user_prompt = f"""
Your task is to summarize news and explain why it matters for children (preschool / early elementary school).

OUTPUT FORMAT (return exactly these ten sections, in this order):
UUID:
Title:
Subtitle: short oneliner in news article style describing the article
Keywords: short descriptive keywords used in the article. Provide no more than 5 frequent keywords separated by commas.
Summary: a concise and succinct summary no longer than 1000 tokens. Always end with a positive take-away.
Language: language the original article was written in e.g. EN = english, DE = german
Source: source and published date in source (YYYY-MM-DD) format based on 'Source' and 'Published_at' E.g. CBC news (2025-11-11)
URL: 
Categories: type of article the news falls under
Score: composite score of educational value and fun factor, range from 1 to 5
Reason: reason for the given score, evaluating educational value and fun factor

STYLE:
- Energetic shōnen-protagonist voice, but professional (no emojis, no slang, no profanity).
- Be factual; do not invent details. If info is missing, say “Insufficient information.”

CONSTRAINTS:
- Do not include anything other than the four labeled sections.
- The example below is illustrative only—do not copy its content or opinions.

SCORING RULES:
- The Score must be based on TWO factors:
  (1) Educational Value: How clearly can a young child learn something concrete from this story?
  (2) Fun Factor: How exciting, adventurous, or imagination-triggering is the story for a child?
- 5 = Extremely educational AND extremely fun
- 4 = Very educational OR very fun (but not both)
- 3 = Decent for learning but not particularly exciting; or exciting but not very educational
- 2 = Low educational content AND not very fun
- 1 = Not appropriate or uninteresting for children

REASON SECTION RULES:
- The Reason must explicitly reference BOTH components: educational value AND fun factor.
- The Reason must directly justify the number (e.g., “This is a 4 because…”).
- Use energetic, encouraging tone (“This story fires up curiosity…”).
- Keep it short (1–2 sentences max).
- NO adult-centric phrases like “might be complex for younger children.”
- NO evaluation of writing quality or political framing.
- NO generic filler (e.g., “This sparks curiosity” without specifics).



Here is an example:
<example>
Input:
UUID: 809a51d5-33c7-4b46-9c5a-7ab7cc7ca6aa
Title: Blue Jays have won the world series
Subtitle: A nation's baseball dream came true after 30 years
Keywords: Blue Jays, Baseball, Encouraging news
Summary: Have you ever dream of being an athelete? Despite being an underdog in the series, the Toronto Blue Jays fough their way through the World Series and knock out the fan favourite choice. Athletes have to train hard to achieve their dreams. The key is to take one day at a time. Let's cheer for the team!
Language: EN
Source: CBC news (2026-11-01)
URL: https://www.cbc.ca/news/politics/second-round-major-projects-nation-building-9.6976532
Categories: sports
Score: 5
Reason: High because it is a historical moment for Canadian baseball fans. Also, who doesn't like a baseball team that is named Blue Jays?
</example>

Now analyze this document:
{document_text}
    """
    
    use_tone="Engaging voice, ELI5 style suitable for preschool / early elementary schoolers (no slang, no profanity)."

    response = client.responses.create(
        model="gpt-4o",
        input=[
            {
                "role": "system",
                "content": (
                    f"""You are a helpful reporter. \n
                    {use_tone} The summary must be ≤1000 tokens.
                    """
                ),
            },
            {"role": "user", "content": user_prompt},
        ],
        max_output_tokens=1000,
    )

    ##return response as text##
    #return response.output_text

    ##return response as Pydantic model##
    output_text = response.output_text.strip()

    # Extract labeled sections using regex
    def extract_field(label):
        pattern = rf"{label}:\s*(.*?)(?=\n[A-Za-z]+:|$)"
        match = re.search(pattern, output_text, re.DOTALL)
        return match.group(1).strip() if match else "Insufficient information."

    # Robust URL extractor (avoids broken regex)
    def extract_url(text):
        url_pattern = r"https?://[^\s]+"
        match = re.search(url_pattern, text)
        return match.group(0) if match else "Insufficient information."

    # Normalize all extracted fields to one line
    def clean_text(t):
        return re.sub(r"\s+", " ", t).strip()

    uuid = extract_field("UUID")
    title = extract_field("Title")
    subtitle = extract_field("Subtitle")
    keywords = extract_field("Keywords")
    summary = extract_field("Summary")
    language = extract_field("Language")
    source = extract_field("Source")
    url = extract_url(output_text)   # note a separate function to extract url  
    categories = extract_field("Categories")
    score_str = extract_field("Score")
    reason = extract_field("Reason")

    # Convert score to int safely
    try:
        score = int(score_str)
    except ValueError:
        score = -1

    # Token usage
    input_tokens = getattr(response.usage, "input_tokens", 0)
    output_tokens = getattr(response.usage, "output_tokens", 0)

    # Clean up extra whitespace
    uuid = clean_text(uuid)
    title = clean_text(title)
    subtitle = clean_text(subtitle)
    keywords = clean_text(keywords)
    summary = clean_text(summary)
    language = clean_text(language)
    source = clean_text(source)
    url = clean_text(url)
    categories = clean_text(categories)
    reason = clean_text(reason)

    # Build and return a Pydantic object
    return SummaryOutput(
        UUID=uuid,
        Title=title,
        Subtitle=subtitle,
        Keywords=keywords,
        Summary=summary,
        Language=language,
        Source=source,
        URL=url,
        Categories=categories,
        Score=score,
        Reason=reason,
        Input_tokens=input_tokens,
        Output_tokens=output_tokens
    )



#scrape website for chromdb in service 2#
from newspaper import Article
import newspaper

# ensure newspaper3k doesn't get blocked
newspaper.Config.HTTPHeaders = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15)'
}

def scrape(url):
    """
    Scrape readable article text using ONLY newspaper3k.
    Returns cleaned article text if available, otherwise an error message.
    """
    try:
        article = Article(url)
        article.download()
        article.parse()

        text = article.text.strip()

        if text:
            return text
        else:
            return "Extraction succeeded but returned empty text."

    except Exception as e:
        return f"newspaper3k failed: {e}"
    



In [33]:
#call OpenAI and save articles db as csv
from pprint import pprint
import pandas as pd

scraped_articles = []

if "data" in data:
    for article in data["data"]:

        # print parsed article
        print("\n=====Article title=====\n")
        print(article["title"])
        print("\n=====Article snippet=====\n")
        print(article["snippet"])

        # run gpt
        result = ask_chatgpt(article)
        print("\n=====GPT result=====\n")
        #print(result)
        #print(result.model_dump_json(indent=2))
        pprint(result.model_dump(), sort_dicts=False)

        # scrape website by url
        full_article=scrape(result.URL)

        scraped_articles.append({
            "uuid": result.UUID,
            "title": result.Title,
            "source": result.Source,
            "url": result.URL,
            "text": full_article
        })

else:
    print("⚠️ Unexpected response structure.")

#export scraped file for service 2
df = pd.DataFrame(scraped_articles)
print("\n=====scraped article result=====\n")
print(scraped_articles)


article_db_filename="scraped_articles.csv"
df.to_csv(f"{article_db_filename}", index=False)



=====Article title=====

JTN Networks Acquires Post Millennial And Human Events In Union Of Brands On The Right

=====Article snippet=====

JTN Networks, the parent company of Just the News, has acquired the Human Events and The Post Millennial, uniting news brands on the right.

Human Events, which...

=====GPT result=====

{'UUID': '948be2f1-bb49-4a84-b931-5cbf24667b10',
 'Title': 'JTN Networks Acquires Post Millennial And Human Events',
 'Subtitle': 'News brands unite on the right.',
 'Keywords': 'JTN Networks, Acquisition, News Brands, Media',
 'Summary': 'Imagine if two teams joined forces to make an even bigger team! '
            "That's what JTN Networks, the company that owns Just the News, "
            'did by bringing together Human Events and The Post Millennial. '
            "These are news brands that share similar ideas, and now they'll "
            "work as one to share stories in a new way. It's like when "
            'superheroes team up to make an awesome super 

In [32]:
print(scraped_articles)

[{'uuid': '948be2f1-bb49-4a84-b931-5cbf24667b10', 'title': 'Big News: JTN Networks Brings Together Two News Teams', 'source': 'deadline.com (2025-11-14)', 'url': 'https://deadline.com/2025/11/right-wing-media-brands-merge-just-the-news-human-events-1236616651/', 'text': 'JTN Networks, the parent company of Just the News, has acquired the Human Events and The Post Millennial, uniting news brands on the right.\n\nHuman Events, which was founded in 1944 and has more recently featured columns and podcasts from Jack Posobiec, the MAGA influencer, will end daily publication this month and will be transformed into a virtual and live events platform, including ticket, music and movie sales and daily event programming, according to JTN Network’s CEO Mark Meckler. John Solomon, the founder of Just the News, will serve as chief strategy and content officer and board chairman.\n\nThe Post Millennial, a Canadian online website, was founded in 2017 but later acquired by Human Events Media Group. Jef